In [7]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from s0_fun_base import XY_grw_nf_compu
import gpflow
from s0_class_GP_IPM import Perted_IPM
from tensorflow_probability import distributions as tfd
f64 = gpflow.utilities.to_default_float

In [ ]:
truedata_style='glm'
grw_setting='sep'
target='grw_nf'
convert_to_constrained_values = 'ON'
if convert_to_constrained_values == 'ON':
    # converting unconstrained values to constrained space & calculating the likelihood values.
    true_population = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/"+truedata_style+"_mle_true_population.pkl", mode="rb"))
    df_grw_nf = XY_grw_nf_compu(true_population)    
    m_grw_nf_new = gpflow.models.GPR(data=(df_grw_nf[0], df_grw_nf[1]), kernel=gpflow.kernels.RBF())
    optimizer = gpflow.optimizers.Scipy()
    optimizer.minimize(m_grw_nf_new.training_loss, m_grw_nf_new.trainable_variables)

    m_grw_nf_new.kernel.lengthscales.prior = tfd.HalfNormal(scale=f64(100.))
    m_grw_nf_new.kernel.variance.prior = tfd.HalfNormal(scale=f64(100.))
    m_grw_nf_new.likelihood.variance.prior = tfd.HalfNormal(scale=f64(100.))


    # We now sample from the posterior using HMC.
    hmc_helper = gpflow.optimizers.SamplingHelper(
        m_grw_nf_new.log_posterior_density, m_grw_nf_new.trainable_parameters
    )

    samples = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/samples_"+target+".pkl", mode="rb")) 
    constrained_samples = hmc_helper.convert_to_constrained_values(samples)
    pickle.dump(constrained_samples, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/constrained_samples_"+target+".pkl", mode="wb"))

    nll = np.array([])
    nlp = np.array([])

    for i in range(5000):
        for var, var_samples in zip(hmc_helper.current_state, samples):
            var.assign(var_samples[i])
        
        nll = np.append(nll, np.array(m_grw_nf_new.log_marginal_likelihood()))
        nlp = np.append(nlp, np.array(m_grw_nf_new.log_posterior_density()))
    
    pickle.dump(nll, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nll_"+target+".pkl", mode="wb"))
    pickle.dump(nlp, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nlp_"+target+".pkl", mode="wb"))

In [8]:
truedata_style='glm'
grw_setting='sep'
target='grw_nf'
opt_percentage = 6
rep = 100
popu_dataset = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/"+truedata_style+"_mle_true_population.pkl", mode="rb"))
models_true = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/gp_"+truedata_style+"mle_mle_models.pkl", mode="rb"))
constrained_samples = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/constrained_samples_"+target+".pkl", mode="rb"))
nlp = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nlp_"+target+".pkl", mode="rb"))

In [10]:
IPM_pret_grw_nf = Perted_IPM(popu_data=popu_dataset, GPmodel_true=models_true, 
                          grw_setting=grw_setting, target='m_' + target, truedata_style=truedata_style,
                          mcmc_para_sample=constrained_samples, summary='Full', nlog_post=nlp, opt_percentage=opt_percentage)

In [13]:
popu_opt_mode = 'OFF'

In [14]:

if popu_opt_mode == 'ON':
    print('\n\n\n' + 'Re-calculating summary_data for the opt MCMC samples' + '\n\n\n')
    summary_opt = IPM_pret_grw_nf.simuVSsimu_singleModel_fun_parallel(rep=rep, random_seed=1,
                                        interested_models='Opt', evaluate_at_training=False)

    pickle.dump(summary_opt, 
                open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_opt_"+target+".pkl", mode="wb"))
    
    summary_around_opt = IPM_pret_grw_nf.simuVSsimu_fun_parallel(rep=rep, random_seed=1, 
                                    opt_percentage=opt_percentage, interested_models='Opt', 
                                    MCMC_boolean_list='Opt', evaluate_at_training=False)
    pickle.dump(summary_around_opt, 
                open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_around_opt_"+target+".pkl", mode="wb"))
    
elif popu_opt_mode == 'OFF':
    print('\n\n\n' + 'Loading summary_data for the opt MCMC samples' + '\n\n\n')
    summary_opt = pickle.load(open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_opt_"+target+".pkl", mode="rb"))
    summary_around_opt = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_around_opt_"+target+".pkl", mode="rb"))




Loading summary_data for the opt MCMC samples





In [19]:
print('\n\n\n' + 'Re-calculating the top summary stats' + '\n\n\n')
num_mcmc = IPM_pret_grw_nf.mcmc_para_sample[0].shape[0]

most_freq_summary_stats = pd.DataFrame(data=0.0, 
                                        index=range(np.sum(IPM_pret_grw_nf.whether_around_opt_comp)), 
                                        columns=IPM_pret_grw_nf.col_names)

for j in range(np.sum(IPM_pret_grw_nf.whether_around_opt_comp)):
    d = summary_around_opt.loc[(0+j*rep):(rep-1+j*rep)].reset_index(drop=True).copy()
    auc0 = np.zeros(33)
    auc1 = np.zeros(33)
    for i in range(33):        
        fpr0, tpr0, _ = roc_curve(y_true=np.append(np.repeat(1, rep), np.repeat(0, rep)), 
                                y_score=np.append(summary_opt.iloc[:, i], d.iloc[:, i]), pos_label=0)
        auc0[i] = auc(fpr0, tpr0)
    
    most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.58]] = most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.58]]+ 1
    





Re-calculating the top summary stats





In [20]:
#0.58
print(np.sort(most_freq_summary_stats.sum())[-10:])
most_columns = most_freq_summary_stats.columns[np.argsort(most_freq_summary_stats.sum())[-10:]]
print(most_columns)

[ 9. 18. 18. 19. 24. 28. 28. 37. 76. 76.]
Index(['27e', '13a', '29g', '22b', '29f', '29c', '29d', '1e', '27d', '27c'], dtype='object')
